In [0]:
# =============================================================================
# ingest-news-doe-sp-spi.ipynb
#
# Diário Oficial do Estado de SP — busca por termo via API JSON (SPI -
# Secretaria de Parcerias em Investimentos, e qualquer outro termo
# adicionado em TERMOS futuramente).
#
# Notebook próprio (não é RSS, "site inteiro", scraping de HTML nem lista
# de PDF -- é busca por termo via API JSON própria do site, mais parecido
# com o antigo notebook de Google News, só que com API estruturada em vez
# de RSS+decodificação de link). Decisão registrada na Fase 1 do teste
# isolado (teste_doe_sp.ipynb).
#
# Validado na Fase 1: 78 itens em janela de 31 dias pro termo "spi", 0 sem
# data, 0 sem slug, 5/5 detalhes obtidos com texto limpo. Volume real é
# maior que qualquer fonte já validada até agora -- por isso o manifesto de
# dedup (mesmo padrão do Acende Brasil/ANTT) é essencial aqui, não
# opcional: rodando diário com janela de 31 dias, a maior parte dos itens
# se repete de execução pra execução.
#
# Achado da Fase 1, importante: a busca é por *menção ao termo*, não só
# publicação *da* secretaria -- um resultado pra "spi" pode ser, por
# exemplo, um extrato de contrato da ARTESP que cita a SPI como poder
# concedente. Isso é esperado, não é erro.
# =============================================================================

In [0]:
%pip install --quiet httpx beautifulsoup4 lxml
dbutils.library.restartPython()

In [0]:
# =============================================================================
# Imports
# =============================================================================

import os
import json
import time
import random
from datetime import datetime, timedelta
from typing import Optional

import httpx
from bs4 import BeautifulSoup

In [0]:
# =============================================================================
# Configuração
# =============================================================================

API_BASE = "https://do-api-web-search.doe.sp.gov.br/v2"

# Lista de termos monitorados -- pode crescer sem precisar de notebook novo
# (é a vantagem principal dessa fonte ser busca por termo, e não uma
# listagem fixa de uma única secretaria).
TERMOS = ["spi"]

SOURCE_ID = "doe_sp_spi"
SOURCE_DESCRICAO = "Diário Oficial do Estado de SP — busca por termo (SPI)"

_HOJE_DT = datetime.today()
HOJE = _HOJE_DT.strftime("%Y-%m-%d")
FROM_DATE = (_HOJE_DT - timedelta(days=31)).strftime("%Y-%-m-%-d")
TO_DATE = _HOJE_DT.strftime("%Y-%-m-%-d")

PAGE_SIZE = 20
HTTP_TIMEOUT = 30

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)

# Segue o padrão corrigido: salva direto em files/{HOJE}, sem subpasta por setor.
PASTA_BASE = f"/Volumes/desafio_kinea/research/research_volume/infraestrutura/files/{HOJE}"
PASTA_MANIFESTOS = "/Volumes/desafio_kinea/research/research_volume/infraestrutura/manifests"
os.makedirs(PASTA_BASE, exist_ok=True)
os.makedirs(PASTA_MANIFESTOS, exist_ok=True)

CAMINHO_MANIFESTO = os.path.join(PASTA_MANIFESTOS, f"{SOURCE_ID}_processados.json")

In [0]:
# =============================================================================
# Carrega a função compartilhada que atualiza o status real da fonte na
# tabela de controle a cada execução. Posicionada aqui, bem antes da
# execução mas depois do restartPython() (nunca entre os dois -- restart
# apaga o que %run já tiver carregado).
# =============================================================================

# MAGIC %run "/Workspace/Shared/Research_Infra/Data-Ingestion-Pipeline-for-Kinea-Research-Infrastructure/scripts/utils_controle"

In [0]:
# =============================================================================
# Manifesto de deduplicação (chave: slug, identificador estável da API)
# =============================================================================

def carregar_manifesto(caminho: str) -> set:
    if not os.path.exists(caminho):
        return set()
    with open(caminho, "r", encoding="utf-8") as f:
        return set(json.load(f))


def salvar_manifesto(caminho: str, processados: set) -> None:
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(sorted(processados), f, ensure_ascii=False, indent=2)

In [0]:
# =============================================================================
# Etapa 1 — Busca por termo (todas as páginas), validado na Fase 1
# =============================================================================

def buscar_publicacoes(termo: str, from_date: str, to_date: str, page_size: int = PAGE_SIZE) -> list[dict]:
    itens = []
    pagina = 1

    while True:
        params = {
            "Terms[0]": termo,
            "FromDate": from_date,
            "ToDate": to_date,
            "PageNumber": pagina,
            "PageSize": page_size,
            "SortField": "Date",
        }
        resp = httpx.get(
            f"{API_BASE}/advanced-search/publications",
            params=params,
            headers={"User-Agent": USER_AGENT},
            timeout=HTTP_TIMEOUT,
        )
        if resp.status_code != 200:
            print(f"  [busca] página {pagina} -> status {resp.status_code}, parando.")
            break

        dados = resp.json()
        pagina_itens = dados.get("items", [])
        itens.extend(pagina_itens)

        print(f"  [busca] página {pagina}/{dados.get('totalPages', '?')}: {len(pagina_itens)} itens.")

        if not dados.get("hasNextPage"):
            break
        pagina += 1
        time.sleep(random.uniform(0.3, 0.8))

    return itens

In [0]:
# =============================================================================
# Etapa 2 — Detalhe de um item + limpeza do HTML do campo `content`
# =============================================================================

def obter_detalhe(slug: str) -> Optional[dict]:
    resp = httpx.get(
        f"{API_BASE}/publications/{slug}",
        headers={"User-Agent": USER_AGENT},
        timeout=HTTP_TIMEOUT,
    )
    if resp.status_code != 200:
        print(f"    -> status {resp.status_code}")
        return None
    return resp.json()


def limpar_html_content(html_bruto: str) -> str:
    soup = BeautifulSoup(html_bruto, "lxml")
    return soup.get_text("\n", strip=True)

In [0]:
# =============================================================================
# Salvamento — mesmo contrato de metadados usado em todo o projeto
# (source_id, title, description, url, date, published_at), com
# publicationType como campo extra (achado da Fase 1 -- carrega
# classificação jurídica útil, ex.: "14.133/21 - Extrato de Convênio").
# =============================================================================

def salvar_artefatos(pasta: str, titulo: str, texto: str, metadados: dict) -> tuple[str, str]:
    import re
    import unicodedata
    import hashlib

    def _slugify(s: str, max_len: int = 60) -> str:
        s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode()
        s = re.sub(r"[^a-zA-Z0-9]+", "-", s).strip("-").lower()
        return (s[:max_len] or "sem-titulo").strip("-")

    nome_base = f"{SOURCE_ID}_{_slugify(titulo)}_{hashlib.md5(metadados['url'].encode()).hexdigest()[:8]}"
    caminho_txt = os.path.join(pasta, f"{nome_base}.txt")
    caminho_json = os.path.join(pasta, f"{nome_base}.json")

    with open(caminho_txt, "w", encoding="utf-8") as f:
        f.write(texto or "")
    with open(caminho_json, "w", encoding="utf-8") as f:
        json.dump(metadados, f, ensure_ascii=False, indent=2)

    return caminho_txt, caminho_json

In [0]:
# =============================================================================
# Execução
# =============================================================================

try:
    ja_processados = carregar_manifesto(CAMINHO_MANIFESTO)

    todos_itens = []
    for termo in TERMOS:
        print(f"Termo: {termo!r}")
        itens_termo = buscar_publicacoes(termo, FROM_DATE, TO_DATE)
        todos_itens.extend(itens_termo)

    print(f"\nTotal de itens encontrados: {len(todos_itens)}")

    # Itens sem data ou sem slug não têm como ser processados de forma
    # confiável -- pula individualmente (igual ao 422 do PSR, mas tratado
    # aqui de forma explícita e não fatal para o restante do lote).
    itens_validos = [i for i in todos_itens if i.get("date") and i.get("slug")]
    ignorados = len(todos_itens) - len(itens_validos)
    if ignorados:
        print(f"[aviso] {ignorados} item(ns) sem data/slug, ignorados.")

    itens_novos = [i for i in itens_validos if i["slug"] not in ja_processados]
    print(f"{len(itens_novos)} itens novos (de {len(itens_validos)} válidos).")

    salvos = 0
    for item in itens_novos:
        detalhe = obter_detalhe(item["slug"])
        if not detalhe:
            print(f"  [ERRO] detalhe não obtido para slug={item['slug']!r}, pulando.")
            continue

        texto_limpo = limpar_html_content(detalhe.get("content", ""))
        url_item = f"https://doe.sp.gov.br/{item['slug']}"

        metadados = {
            "source_id": SOURCE_ID,
            "title": detalhe.get("title") or item.get("title"),
            "description": SOURCE_DESCRICAO,
            "url": url_item,
            "date": detalhe.get("date") or item.get("date"),
            "published_at": detalhe.get("date") or item.get("date"),
            "publicationType": detalhe.get("publicationType"),
            "journal": detalhe.get("journal"),
            "section": detalhe.get("section"),
        }

        salvar_artefatos(PASTA_BASE, metadados["title"] or "sem-titulo", texto_limpo, metadados)
        ja_processados.add(item["slug"])
        salvos += 1
        time.sleep(random.uniform(0.3, 0.8))

    salvar_manifesto(CAMINHO_MANIFESTO, ja_processados)
    print(f"\n=== Fim. {salvos} novo(s) salvo(s) em {PASTA_BASE} ===")

    atualizar_status_fonte(
        source_id=SOURCE_ID,
        sucesso=True,
        docs_capturados=salvos,
    )

except Exception as e:
    print(f"\n=== ERRO GERAL: {e} ===")
    atualizar_status_fonte(
        source_id=SOURCE_ID,
        sucesso=False,
        docs_capturados=0,
        erro=str(e),
    )